In [1]:
from pathlib import Path
import json

import numpy as np
import pandas as pd

import faiss
from sentence_transformers import SentenceTransformer

/home/nineleaps/client_meeting_agent/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
from pathlib import Path

DATA_DIR = Path("data")

documents = []

for file_path in DATA_DIR.rglob("*.md"):
    text = file_path.read_text(encoding="utf-8")

    documents.append({
        "text": text,
        "source": str(file_path),
        "document_type": file_path.parent.name
    })

print(f"Total documents loaded: {len(documents)}")

Total documents loaded: 9


In [6]:
print("Number of documents:", len(documents))

print("\nDocument details:\n")

for i, doc in enumerate(documents):
    print(f"Document {i + 1}")
    print(f"Source: {doc['source']}")
    print(f"Type: {doc['document_type']}")
    print(f"Characters: {len(doc['text'])}")
    print("-" * 60)

Number of documents: 9

Document details:

Document 1
Source: data/action_items/acme_open_action_items.md
Type: action_items
Characters: 844
------------------------------------------------------------
Document 2
Source: data/action_items/acme_account_strategy.md
Type: action_items
Characters: 834
------------------------------------------------------------
Document 3
Source: data/meeting_notes/2026-08-14_acme_executive_checkin.md
Type: meeting_notes
Characters: 839
------------------------------------------------------------
Document 4
Source: data/meeting_notes/2026-06-18_acme_qbr.md
Type: meeting_notes
Characters: 863
------------------------------------------------------------
Document 5
Source: data/meeting_notes/2026-07-22_acme_technical_review.md
Type: meeting_notes
Characters: 812
------------------------------------------------------------
Document 6
Source: data/emails/2026-08-25_acme_internal_followup.md
Type: emails
Characters: 600
------------------------------------------

In [7]:
print(documents[0]["text"])

# Acme Corp — Open Action Items

| ID | Action Item | Owner | Due Date | Status | Priority |
|---|---|---|---|---|---|
| A-001 | Send authentication configuration guide | Solutions Team | 2026-08-28 | Open | High |
| A-002 | Confirm sandbox access | Engineering Team | 2026-08-29 | Open | High |
| A-003 | Provide API rate-limit documentation | Technical Documentation | 2026-08-30 | Open | Medium |
| A-004 | Prepare ROI summary for leadership | Account Team | 2026-08-31 | In Progress | High |
| A-005 | Propose timeline for remaining integration work | Solutions Team | 2026-09-02 | Open | High |
| A-006 | Recommend whether Acme is ready for expansion | Account Manager | 2026-09-03 | Open | High |

## Important
The upcoming meeting should explicitly review overdue and high-priority items, identify blockers, and assign owners and dates.



In [8]:
df_documents = pd.DataFrame(documents)

df_documents

,text,source,document_type
0,# Acme Corp — Open Action Items\n\n| ID | Acti...,data/action_items/acme_open_action_items.md,action_items
1,# Acme Corp — Account Strategy Notes\n\n## Rel...,data/action_items/acme_account_strategy.md,action_items
2,# Acme Corp — Executive Check-in\nDate: 2026-0...,data/meeting_notes/2026-08-14_acme_executive_c...,meeting_notes
3,# Acme Corp — Quarterly Business Review\nDate:...,data/meeting_notes/2026-06-18_acme_qbr.md,meeting_notes
4,# Acme Corp — Technical Review\nDate: 2026-07-...,data/meeting_notes/2026-07-22_acme_technical_r...,meeting_notes
5,# Internal Account Email\nDate: 2026-08-25\nFr...,data/emails/2026-08-25_acme_internal_followup.md,emails
6,# Email — Acme Follow-up\nDate: 2026-08-20\nFr...,data/emails/2026-08-20_acme_followup.md,emails
7,# Acme Corp — Business Overview\n\nAcme Corp o...,data/client_documents/acme_corp_business_overv...,client_documents
8,# Acme Corp — Client Profile\n\n## Company\nAc...,data/client_documents/acme_corp_profile.md,client_documents


In [10]:
df_documents["document_type"].value_counts()

document_type
meeting_notes       3
action_items        2
emails              2
client_documents    2
Name: count, dtype: int64

In [11]:
CHUNK_SIZE = 500
CHUNK_OVERLAP = 50

chunks = []

for doc in documents:
    text = doc["text"]

    start = 0

    while start < len(text):
        end = start + CHUNK_SIZE

        chunk_text = text[start:end]

        chunks.append({
            "text": chunk_text,
            "source": doc["source"],
            "document_type": doc["document_type"]
        })

        start += CHUNK_SIZE - CHUNK_OVERLAP

print(f"Total chunks created: {len(chunks)}")

Total chunks created: 19


In [12]:
for i, chunk in enumerate(chunks[:5]):
    print(f"\n--- Chunk {i + 1} ---")
    print(f"Source: {chunk['source']}")
    print(f"Type: {chunk['document_type']}")
    print(f"Characters: {len(chunk['text'])}")
    print(chunk["text"])


--- Chunk 1 ---
Source: data/action_items/acme_open_action_items.md
Type: action_items
Characters: 500
# Acme Corp — Open Action Items

| ID | Action Item | Owner | Due Date | Status | Priority |
|---|---|---|---|---|---|
| A-001 | Send authentication configuration guide | Solutions Team | 2026-08-28 | Open | High |
| A-002 | Confirm sandbox access | Engineering Team | 2026-08-29 | Open | High |
| A-003 | Provide API rate-limit documentation | Technical Documentation | 2026-08-30 | Open | Medium |
| A-004 | Prepare ROI summary for leadership | Account Team | 2026-08-31 | In Progress | High |
| A-

--- Chunk 2 ---
Source: data/action_items/acme_open_action_items.md
Type: action_items
Characters: 394
ount Team | 2026-08-31 | In Progress | High |
| A-005 | Propose timeline for remaining integration work | Solutions Team | 2026-09-02 | Open | High |
| A-006 | Recommend whether Acme is ready for expansion | Account Manager | 2026-09-03 | Open | High |

## Important
The upcoming meeting sho

In [13]:
df_chunks = pd.DataFrame(chunks)

df_chunks["document_type"].value_counts()

document_type
meeting_notes       6
client_documents    5
action_items        4
emails              4
Name: count, dtype: int64

In [14]:
EMBEDDING_MODEL_NAME = "all-MiniLM-L6-v2"

embedding_model = SentenceTransformer(EMBEDDING_MODEL_NAME)

print("Embedding model loaded successfully!")

'[SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: self-signed certificate in certificate chain (_ssl.c:1000)' thrown while requesting HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/adapter_config.json
Retrying in 1s [Retry 1/5].
'[SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: self-signed certificate in certificate chain (_ssl.c:1000)' thrown while requesting HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/adapter_config.json
Retrying in 2s [Retry 2/5].
'[SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: self-signed certificate in certificate chain (_ssl.c:1000)' thrown while requesting HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/adapter_config.json
Retrying in 4s [Retry 3/5].
'[SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: self-signed certificate in certificate chain (_ssl.c:1000)' thrown while requesting HEAD https://huggingface.co/sentence

'[SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: self-signed certificate in certificate chain (_ssl.c:1000)' thrown while requesting HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/2_Normalize/config.json
Retrying in 1s [Retry 1/5].
'[SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: self-signed certificate in certificate chain (_ssl.c:1000)' thrown while requesting HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/2_Normalize/config.json
Retrying in 2s [Retry 2/5].
'[SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: self-signed certificate in certificate chain (_ssl.c:1000)' thrown while requesting HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/2_Normalize/config.json
Retrying in 4s [Retry 3/5].
'[SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: self-signed certificate in certificate chain (_ssl.c:1000)' thrown while requesting HEAD https://huggingface

Embedding model loaded successfully!


In [15]:
chunk_texts = [chunk["text"] for chunk in chunks]

embeddings = embedding_model.encode(
    chunk_texts,
    convert_to_numpy=True,
    show_progress_bar=True
)

print("Embedding shape:", embeddings.shape)

Batches: 100%|███████████████████████████████████| 1/1 [00:01<00:00,  1.36s/it]

Embedding shape: (19, 384)


In [16]:
print("First embedding:")
print(embeddings[0])

First embedding:
[-2.13950370e-02 -1.42965065e-02 -8.36147070e-02  2.14530919e-02
 -1.72184743e-02  5.04304022e-02 -5.43252341e-02  1.31677585e-02
 -1.17823863e-02  8.67104810e-03  4.45861630e-02 -2.02070717e-02
  2.84047890e-02 -6.13669641e-02  3.82520109e-02  3.45811211e-02
 -2.24717986e-02 -4.05911244e-02 -2.33320948e-02 -1.27352610e-01
  1.99202597e-02  4.32715844e-03 -3.08214705e-02 -6.85951039e-02
 -5.85773140e-02 -3.02403793e-02 -1.20828385e-02  1.53421778e-02
 -5.15417606e-02 -4.70241234e-02 -1.51742473e-02  1.93660408e-02
  5.32207973e-02  4.45095152e-02  6.09484687e-02  6.39471337e-02
 -1.52795399e-02 -4.70774621e-02  9.60902423e-02 -6.53982013e-02
 -3.83132324e-03 -1.21023402e-01 -2.37821154e-02 -2.86900601e-03
 -1.00940140e-02 -1.65274311e-02 -6.79988265e-02 -5.54938987e-02
  1.16940318e-02  7.40415901e-02 -5.54685108e-02 -5.38101643e-02
 -1.03237852e-02 -7.10409740e-03 -1.78529713e-02  3.60224359e-02
 -6.40743747e-02 -4.14147004e-02 -2.39299238e-02 -1.06791943e-01
  5.2157

In [17]:
print("Number of chunks:", len(chunks))
print("Number of embeddings:", len(embeddings))
print("Embedding dimensions:", embeddings.shape[1])

Number of chunks: 19
Number of embeddings: 19
Embedding dimensions: 384


In [18]:
embeddings = embeddings.astype("float32")

print("Embedding data type:", embeddings.dtype)
print("Embedding shape:", embeddings.shape)

Embedding data type: float32
Embedding shape: (19, 384)


In [19]:
embedding_dimension = embeddings.shape[1]

index = faiss.IndexFlatL2(embedding_dimension)

print("FAISS index created!")
print("Embedding dimension:", embedding_dimension)

FAISS index created!
Embedding dimension: 384


In [20]:
index.add(embeddings)

print("Number of vectors in FAISS:", index.ntotal)

Number of vectors in FAISS: 19


In [21]:
print("Number of chunks:", len(chunks))
print("Number of embeddings:", len(embeddings))
print("Number of FAISS vectors:", index.ntotal)

Number of chunks: 19
Number of embeddings: 19
Number of FAISS vectors: 19


In [22]:
print(index)

<faiss.swigfaiss.IndexFlatL2; proxy of <Swig Object of type 'faiss::IndexFlatL2 *' at 0x7666de23fc70> >


In [24]:
VECTOR_STORE_DIR = Path("vector_store")
VECTOR_STORE_DIR.mkdir(parents=True, exist_ok=True)

print("Vector store directory:", VECTOR_STORE_DIR.resolve())

Vector store directory: /home/nineleaps/client_meeting_agent/vector_store


In [25]:
faiss_index_path = VECTOR_STORE_DIR / "acme_documents.index"

faiss.write_index(
    index,
    str(faiss_index_path)
)

print("FAISS index saved to:")
print(faiss_index_path.resolve())

FAISS index saved to:
/home/nineleaps/client_meeting_agent/vector_store/acme_documents.index


In [26]:
metadata_path = VECTOR_STORE_DIR / "metadata.json"

with open(metadata_path, "w", encoding="utf-8") as f:
    json.dump(
        chunks,
        f,
        indent=2,
        ensure_ascii=False
    )

print("Metadata saved to:")
print(metadata_path.resolve())

Metadata saved to:
/home/nineleaps/client_meeting_agent/vector_store/metadata.json


In [27]:
print("Files in vector_store:")

for file in VECTOR_STORE_DIR.iterdir():
    print(f"- {file.name}")

Files in vector_store:
- metadata.json
- acme_documents.index


In [28]:
with open(metadata_path, "r", encoding="utf-8") as f:
    saved_chunks = json.load(f)

print("Saved chunks:", len(saved_chunks))
print("Original chunks:", len(chunks))

Saved chunks: 19
Original chunks: 19


In [29]:
loaded_index = faiss.read_index(
    str(faiss_index_path)
)

print("Loaded FAISS vectors:", loaded_index.ntotal)
print("Vector dimension:", loaded_index.d)

Loaded FAISS vectors: 19
Vector dimension: 384


In [30]:
import json
import faiss

VECTOR_STORE_DIR = Path("vector_store")

faiss_index_path = VECTOR_STORE_DIR / "acme_documents.index"
metadata_path = VECTOR_STORE_DIR / "metadata.json"

loaded_index = faiss.read_index(str(faiss_index_path))

with open(metadata_path, "r", encoding="utf-8") as f:
    saved_chunks = json.load(f)

print("FAISS vectors:", loaded_index.ntotal)
print("Metadata records:", len(saved_chunks))

FAISS vectors: 19
Metadata records: 19


In [31]:
query = "What are Acme Corp's current integration blockers?"

print(query)

What are Acme Corp's current integration blockers?


In [32]:
query_embedding = embedding_model.encode(
    [query],
    convert_to_numpy=True
).astype("float32")

print("Query embedding shape:", query_embedding.shape)

Query embedding shape: (1, 384)


In [33]:
k = 5

distances, indices = loaded_index.search(
    query_embedding,
    k
)

print("Indices:")
print(indices)

print("\nDistances:")
print(distances)

Indices:
[[ 2  8 16  1 14]]

Distances:
[[0.5956421  0.6305757  0.88170207 0.9068695  0.9300886 ]]


In [34]:
for rank, (idx, distance) in enumerate(
    zip(indices[0], distances[0]),
    start=1
):

    chunk = saved_chunks[idx]

    print(f"\n{'=' * 70}")
    print(f"Rank: {rank}")
    print(f"Distance: {distance:.4f}")
    print(f"Source: {chunk['source']}")
    print(f"Document Type: {chunk['document_type']}")
    print("\nText:")
    print(chunk["text"])


Rank: 1
Distance: 0.5956
Source: data/action_items/acme_account_strategy.md
Document Type: action_items

Text:
# Acme Corp — Account Strategy Notes

## Relationship Goal
Retain Acme as a strategic enterprise customer and expand usage after the current implementation concerns are addressed.

## Near-Term Goal
Demonstrate that the remaining integration blockers are controlled and that the existing deployment is delivering measurable business value.

## Expansion Trigger
Acme is more likely to approve expansion when:
- ROI is clearly documented.
- Technical blockers have owners and dates.
- API integration 

Rank: 2
Distance: 0.6306
Source: data/meeting_notes/2026-07-22_acme_technical_review.md
Document Type: meeting_notes

Text:
# Acme Corp — Technical Review
Date: 2026-07-22

## Attendees
- Daniel Wong — Director of Engineering, Acme Corp
- Priya Sharma — Account Manager
- Alex Rao — Solutions Consultant

## Discussion
- Daniel said the main integration blocker is authentication config

In [36]:
def search_documents(query, k=5):

    query_embedding = embedding_model.encode(
        [query],
        convert_to_numpy=True
    ).astype("float32")

    distances, indices = loaded_index.search(
        query_embedding,
        k
    )

    results = []

    for idx, distance in zip(indices[0], distances[0]):

        result = saved_chunks[idx].copy()

        result["distance"] = float(distance)

        results.append(result)

    return results

In [37]:
results = search_documents(
    "What are Acme Corp's current integration blockers?",
    k=5
)

In [38]:
for i, result in enumerate(results, start=1):

    print(f"\n{'=' * 70}")
    print(f"Result {i}")
    print(f"Distance: {result['distance']:.4f}")
    print(f"Source: {result['source']}")
    print(f"Type: {result['document_type']}")
    print("\n", result["text"])
    


Result 1
Distance: 0.5956
Source: data/action_items/acme_account_strategy.md
Type: action_items

 # Acme Corp — Account Strategy Notes

## Relationship Goal
Retain Acme as a strategic enterprise customer and expand usage after the current implementation concerns are addressed.

## Near-Term Goal
Demonstrate that the remaining integration blockers are controlled and that the existing deployment is delivering measurable business value.

## Expansion Trigger
Acme is more likely to approve expansion when:
- ROI is clearly documented.
- Technical blockers have owners and dates.
- API integration 

Result 2
Distance: 0.6306
Source: data/meeting_notes/2026-07-22_acme_technical_review.md
Type: meeting_notes

 # Acme Corp — Technical Review
Date: 2026-07-22

## Attendees
- Daniel Wong — Director of Engineering, Acme Corp
- Priya Sharma — Account Manager
- Alex Rao — Solutions Consultant

## Discussion
- Daniel said the main integration blocker is authentication configuration.
- The client need